In [3]:
import os
import json
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import ast
import pandas as pd
import string
import random

import numpy as np

In [4]:
COLOR_MAPPING = {
    (0xFF, 0xD7, 0x00): 'common room',
    (0xFF, 0xA5, 0x00): 'master room',
    (0xEE, 0xE8, 0xAA): 'living room',
    (0x6B, 0x8E, 0x23): 'balcony',
    (0xAD, 0xD8, 0xE6): 'bathroom',
    (0xF0, 0x80, 0x80): 'kitchen',
    (0xDD, 0xA0, 0xDD): 'storage',
    (0xDA, 0x70, 0xD6): 'dining',
}

In [ ]:
# Paths
base_dir = Path("../..")
# json_path = "groups_same_first.json"
csv_path = "groups_same_first_global_results.csv"
img_dir = base_dir / "data" / "floorplan_image"
txt_dir = base_dir / "annotation" / "human_annotated_tags"
out_dir = Path(".") / "examples_output"
out_dir.mkdir(exist_ok=True)

# Load JSON entries
# with open(json_path, 'r') as f:
#     entries = json.load(f)

# Prepare font for subtitles
try:
    font = ImageFont.truetype("arial.ttf", size=16)
except IOError:
    font = ImageFont.load_default()

subtitle_height = 20  # space for ID subtitles

df = pd.read_csv(csv_path)
entries = df['options'].apply(lambda s: ast.literal_eval(s))

for idx, entry in enumerate(entries, start=1):
    group_ids = entry
    images = []

    # Load images for this group
    for img_id in group_ids:
        img_path = img_dir / f"{img_id}.png"
        if img_path.exists():
            images.append((img_id, Image.open(img_path)))
        else:
            raise FileNotFoundError(f"Image file not found: {img_path}")

    # Determine layout
    widths, heights = zip(*(im.size for _, im in images))
    total_width = sum(widths)
    max_height = max(heights)

    # Create canvas
    canvas = Image.new('RGB', (total_width, max_height + subtitle_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(canvas)

    # Paste images and draw subtitles
    x_offset = 0
    for img_id, im in images:
        canvas.paste(im, (x_offset, 0))
        # Compute text size
        if hasattr(font, 'getsize'):
            text_w, text_h = font.getsize(img_id)
        else:
            bbox = draw.textbbox((0,0), img_id, font=font)
            text_w, text_h = bbox[2] - bbox[0], bbox[3] - bbox[1]
        text_x = x_offset + (im.width - text_w) // 2
        text_y = max_height + (subtitle_height - text_h) // 2
        draw.text((text_x, text_y), img_id, fill=(0, 0, 0), font=font)
        x_offset += im.width

    # Save combined image
    img_out_path = out_dir / f"example_{idx}_2.png"
    canvas.save(img_out_path)

    # # Collect and write text descriptions
    # txt_out_path = out_dir / f"example_{idx}.txt"
    # with open(txt_out_path, 'w') as txt_out:
    #     for img_id, _ in images:
    #         tag_file = txt_dir / f"{img_id}.txt"
    #         if tag_file.exists():
    #             with open(tag_file, 'r') as tf:
    #                 desc = tf.read().strip()
    #         else:
    #             raise FileNotFoundError(f"Tag file not found: {tag_file}")
    #         txt_out.write(f"ID: {img_id}, description: {desc}\n\n")

print(f"Generated {len(entries)} example image/text pairs in '{out_dir.resolve()}'")



FileNotFoundError: Image file not found: ../../data/floorplan_image/dataset.png

In [4]:
import glob

In [36]:
input_folder  = 'examples_same_second'
output_folder = os.path.join(input_folder, 'combined')
os.makedirs(output_folder, exist_ok=True)

# Find every file that ends with '_2.png'
pattern = os.path.join(input_folder, '*_2.png')
for suffix_path in glob.glob(pattern):
    # Derive the base filename by stripping off '_2' before the extension
    folder, suffix_fn = os.path.split(suffix_path)
    stem = suffix_fn[:-6]  # removes the trailing "_2.png"
    base_fn = stem + '.png'
    base_path = os.path.join(folder, base_fn)

    if not os.path.exists(base_path):
        print(f"No base image found for {suffix_fn}, skipping.")
        continue

    # Open both images (they must be same size)
    img_base   = Image.open(base_path)
    img_suffix = Image.open(suffix_path)
    w, h       = img_base.size

    # Stack base on top of suffix
    combined = Image.new('RGB', (w, 2*h))
    combined.paste(img_base,   (0, 0))
    combined.paste(img_suffix, (0, h))

    # Save
    out_fn = f"{stem}_combined.png"
    combined.save(os.path.join(output_folder, out_fn))



No base image found for example_2.png, skipping.


In [8]:
base_dir = Path("../..")
csv_path = "difficult_dataset_mid.csv"
img_dir = base_dir / "data" / "floorplan_image"
reoriented_img_dir = base_dir / "data" / "floorplan_reoriented"
out_dir = Path(".") / "examples_output"
out_dir.mkdir(exist_ok=True)

# Try to load a scalable font at 24px
font_size = 24
for font_name in ("arial.ttf", "DejaVuSans.ttf"):
    try:
        font = ImageFont.truetype(font_name, size=font_size)
        break
    except IOError:
        font = None
if font is None:
    # last resort: default (will ignore size)
    font = ImageFont.load_default()
    print("Warning: falling back to default font; letter size may not change.")

subtitle_height = font_size + 8  # give a bit of padding

# Read groups
df = pd.read_csv(csv_path)
entries = df['options'].apply(lambda s: ast.literal_eval(s))

# Prepare A, B, C… labels
letters = list(string.ascii_uppercase)

for idx, entry in enumerate(entries, start=1):
    images = []

    # Load images for this group
    for img_id in entry:
        is_reoriented = random.choice([0, 1])

        if is_reoriented:
            # img_path = reoriented_img_dir / f"{img_id}.png"
            img_path = img_dir / f"{img_id}.png"
        else:
            img_path = img_dir / f"{img_id}.png"
            # img_path = reoriented_img_dir / f"{img_id}.png"
        if not img_path.exists():
            raise FileNotFoundError(f"Image file not found: {img_path}")
        images.append(Image.open(img_path))

    # Compute canvas size
    widths, heights = zip(*(im.size for im in images))
    total_width = sum(widths)
    max_height = max(heights)

    # Create canvas
    canvas = Image.new("RGB", (total_width, max_height + subtitle_height), "white")
    draw = ImageDraw.Draw(canvas)

    # Paste & subtitle
    x = 0
    for i, im in enumerate(images):
        canvas.paste(im, (x, 0))
        label = letters[i]
        # measure
        if hasattr(font, "getsize"):
            w, h = font.getsize(label)
        else:
            bx0, by0, bx1, by1 = draw.textbbox((0,0), label, font=font)
            w, h = bx1 - bx0, by1 - by0
        tx = x + (im.width - w) // 2
        ty = max_height + (subtitle_height - h) // 2
        draw.text((tx, ty), label, fill="black", font=font)
        x += im.width

    # Save
    canvas.save(out_dir / f"example_{idx}_2.png")

print(f"Generated {len(entries)} example image groups in '{out_dir.resolve()}'")

Generated 5000 example image groups in '/home/airlay88/planscape/examples/complex/examples_output'


In [7]:
def add_legend_tight(
    image: Image.Image,
    mapping: dict,
    legend_gap: int = 75,
    font_size: int = 16
) -> Image.Image:
    swatch  = 20
    pad_x   = 5
    pad_y   = 2
    pad_mid = legend_gap

    # try to load a truetype font at the requested size, fallback to default
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", font_size)
    except IOError:
        font = ImageFont.load_default()

    entries = list(mapping.items())

    # measure text widths
    dummy     = Image.new("RGB", (1,1))
    draw0     = ImageDraw.Draw(dummy)
    text_widths = [
        draw0.textbbox((0,0), label, font=font)[2]
        for _, label in entries
    ]

    # compute legend dims
    entry_widths   = [swatch + pad_x + w for w in text_widths]
    total_legend_w = sum(entry_widths) + pad_x * (len(entries) + 1)
    legend_h       = swatch + 2 * pad_y

    # render legend
    legend = Image.new("RGB", (total_legend_w, legend_h), "white")
    draw   = ImageDraw.Draw(legend)

    x = pad_x
    for (color, label), w in zip(entries, text_widths):
        draw.rectangle([x, pad_y, x + swatch, pad_y + swatch],
                       fill=color, outline="black")
        draw.text((x + swatch + pad_x, pad_y),
                  label, fill="black", font=font)
        x += swatch + pad_x + w + pad_x

    # composite under the image
    out_w = max(image.width, total_legend_w)
    out_h = image.height + pad_mid + legend_h
    combined = Image.new("RGB", (out_w, out_h), "white")
    combined.paste(image, ((out_w - image.width)//2, 0))
    combined.paste(legend, ((out_w - total_legend_w)//2, image.height + pad_mid))

    return combined

In [9]:
from PIL import Image, ImageDraw, ImageFont

def add_legend_tight_boxed(
    image: Image.Image,
    mapping: dict,
    legend_gap: int = 75,
    font_size: int = 16,
    title: str = "Legend",
    title_font_size: int = None,
    box_pad: int = 8,              # padding inside the box
    title_gap: int = 6,            # space between title and entries
    box_border: int = 2,           # border thickness
    box_bg: str = "white",
    box_border_color: str = "black",
    round_radius: int = 10         # set to 0 for square corners
) -> Image.Image:
    """
    Renders a single-row legend wrapped in a labeled box under the image.
    """

    swatch  = 20
    pad_x   = 5
    pad_y   = 2
    pad_mid = legend_gap

    # fonts
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", font_size)
        tfont = ImageFont.truetype("DejaVuSans.ttf", title_font_size or (font_size + 2))
    except IOError:
        font = ImageFont.load_default()
        tfont = ImageFont.load_default()

    entries = list(mapping.items())

    # --- measure text widths for entries ---
    dummy = Image.new("RGB", (1, 1))
    d0 = ImageDraw.Draw(dummy)

    text_widths = [d0.textbbox((0, 0), label, font=font)[2] for _, label in entries]
    entry_widths   = [swatch + pad_x + w for w in text_widths]
    total_legend_w = sum(entry_widths) + pad_x * (len(entries) + 1)
    legend_h       = swatch + 2 * pad_y

    # --- measure title ---
    title_bbox = d0.textbbox((0, 0), title, font=tfont)
    title_w = title_bbox[2] - title_bbox[0]
    title_h = title_bbox[3] - title_bbox[1]

    # --- compute box dims ---
    inner_w = max(total_legend_w, title_w)
    inner_h = title_h + title_gap + legend_h
    box_w   = inner_w + 2 * box_pad
    box_h   = inner_h + 2 * box_pad

    # --- render boxed legend ---
    panel = Image.new("RGB", (box_w, box_h), box_bg)
    draw  = ImageDraw.Draw(panel)

    # border (rounded if available)
    rect_xy = [0.5, 0.5, box_w - 0.5, box_h - 0.5]  # half-pixel aligns stroke nicely
    if round_radius > 0 and hasattr(draw, "rounded_rectangle"):
        draw.rounded_rectangle(rect_xy, radius=round_radius, outline=box_border_color, width=box_border, fill=box_bg)
    else:
        draw.rectangle(rect_xy, outline=box_border_color, width=box_border, fill=box_bg)

    # title (centered)
    title_x = (box_w - title_w) // 2
    title_y = box_pad
    draw.text((title_x, title_y), title, fill="black", font=tfont)

    # entries row
    x = (box_w - total_legend_w) // 2
    y_top = box_pad + title_h + title_gap
    for (color, label), w in zip(entries, text_widths):
        # swatch
        draw.rectangle([x, y_top + pad_y, x + swatch, y_top + pad_y + swatch],
                       fill=color, outline="black")
        # label
        draw.text((x + swatch + pad_x, y_top + pad_y), label, fill="black", font=font)
        x += swatch + pad_x + w + pad_x

    # --- composite under the image ---
    out_w = max(image.width, box_w)
    out_h = image.height + pad_mid + box_h
    combined = Image.new("RGB", (out_w, out_h), "white")
    combined.paste(image, ((out_w - image.width) // 2, 0))
    combined.paste(panel, ((out_w - box_w) // 2, image.height + pad_mid))

    return combined


In [5]:
def draw_5x256_boxes(image, box_color="gray", box_width=3):
    """
    Draws five 256x256 gray boxes at (0,0), (256,0), (512,0), (768,0), (1024,0).
    Designed for 1280x288 images where the bottom ~32px holds labels.
    """
    W, H = image.size
    n, s = 5, 256
    total = n * s
    x0 = 0 if W == total else max(0, (W - total) // 2)
    y0 = 0  # tiles start at the top

    out = image.copy()
    draw = ImageDraw.Draw(out)

    for i in range(n):
        x = x0 + i * s
        y = y0
        draw.rectangle([x, y, x + s, y + s], outline=box_color, width=box_width)

    return out

In [9]:
in_dir = "./randomized_options/examples_diff_first"
out_dir = "./randomized_options/examples_diff_first_wlegend_onlyoptionbound"

In [4]:
for fn in os.listdir(in_dir):
    img_path = os.path.join(in_dir, fn)
    
    im = add_legend_tight_boxed(Image.open(img_path), COLOR_MAPPING, title="Color-Coding of Room Types")
    output_path = os.path.join(out_dir, fn)

    im.save(output_path, format="PNG")

NameError: name 'add_legend_tight_boxed' is not defined

In [10]:
for fn in os.listdir(in_dir):
    img_path = os.path.join(in_dir, fn)
    im = Image.open(img_path)

    # draw the 5 fixed 256×256 boxes
    im = draw_5x256_boxes(im)

    # add your boxed legend
    # im = add_legend_tight_boxed(im, COLOR_MAPPING, title="Color-Coding of Room Types")

    output_path = os.path.join(out_dir, fn)
    im.save(output_path, format="PNG")

In [74]:
def first_int(cell):
    """Parse first int from values like '[3, 17]' or '\"[3, 17]\"' or actual lists."""
    v = cell
    # Strip outer quotes like "\"[3, 17]\"" if present
    if isinstance(v, str):
        s = v.strip()
        if (s.startswith('"') and s.endswith('"')) or (s.startswith("'") and s.endswith("'")):
            s = s[1:-1]
        v = s
    if isinstance(v, str):
        v = ast.literal_eval(v)  # parse to list
    return int(v[0])

def train_test_split_by_unordered_pair(
    df,
    base_col="base_cluster",
    other_col="other_cluster",
    test_size=0.2,
    seed=42,
    keep_pair_col=True,
    orig_index_col="orig_index"
):
    """
    Splits so unordered pair {first(base), first(other)} stays in exactly one split.
    Adds a column with the original DataFrame index so you can trace each row back.
    """
    df = df.copy()

    # Preserve original row index for traceability
    # (won't overwrite if the column already exists)
    if orig_index_col in df.columns:
        # Make a unique name if needed
        i = 1
        base_name = orig_index_col
        while orig_index_col in df.columns:
            orig_index_col = f"{base_name}_{i}"
            i += 1
    df[orig_index_col] = df.index

    # --- build unordered pair key: "{min(b_first, o_first)}_{max(b_first, o_first)}"
    b_first = df[base_col].apply(first_int).to_numpy()
    o_first = df[other_col].apply(first_int).to_numpy()
    low  = np.minimum(b_first, o_first)
    high = np.maximum(b_first, o_first)
    df["_pair"] = (low.astype(str) + "_" + high.astype(str))  # unordered key

    # count rows per unordered pair
    counts = df["_pair"].value_counts()

    # greedy group-wise selection for TEST to hit ~test_size rows
    rnd = random.Random(seed)
    pairs = list(counts.index)
    rnd.shuffle(pairs)

    target_test = round(test_size * len(df))
    test_pairs, test_rows = set(), 0
    for p in pairs:
        gain = counts[p]
        if abs((test_rows + gain) - target_test) <= abs(test_rows - target_test):
            test_pairs.add(p)
            test_rows += gain

    # ensure both sides non-empty if possible
    if test_rows == 0 and len(pairs) > 0:
        smallest = counts.sort_values().index[0]
        test_pairs.add(smallest)
        test_rows += counts[smallest]

    test_df  = df[df["_pair"].isin(test_pairs)].copy()
    train_df = df[~df["_pair"].isin(test_pairs)].copy()

    # safety: no unordered-pair overlap
    assert set(train_df["_pair"]).isdisjoint(set(test_df["_pair"])), "Pair leakage detected."

    if not keep_pair_col:
        train_df.drop(columns=["_pair"], inplace=True)
        test_df.drop(columns=["_pair"], inplace=True)

    # Clean row indices but keep the original in `orig_index_col`
    train_df.reset_index(drop=True, inplace=True)
    test_df.reset_index(drop=True, inplace=True)

    return train_df, test_df

In [79]:
df = pd.read_csv("easy_dataset_mid.csv")
train_df, test_df = train_test_split_by_unordered_pair(df, test_size=0.2, seed=123)


In [80]:
train_df['split'] = 'train'
test_df['split'] = 'test'

In [81]:
train_df

,base_cluster,other_cluster,group,outlier_id,options,orig_index,_pair,split
0,"[3, 17]","[19, 18]","['64047', '38478', '33084', '54792', '41277']",41277,"['33084', '38478', '64047', '41277', '54792']",0,3_19,train
1,"[19, 13]","[9, 11]","['42612', '35479', '43396', '26354', '335']",335,"['43396', '35479', '26354', '335', '42612']",2,9_19,train
2,"[19, 10]","[9, 26]","['25168', '73023', '21540', '55791', '50181']",50181,"['55791', '50181', '25168', '21540', '73023']",4,9_19,train
3,"[0, 2]","[3, 28]","['59390', '67842', '80744', '44125', '42226']",42226,"['59390', '67842', '44125', '80744', '42226']",5,0_3,train
4,"[19, 5]","[3, 7]","['72346', '63902', '55314', '59952', '4286']",4286,"['59952', '55314', '63902', '72346', '4286']",6,3_19,train
...,...,...,...,...,...,...,...,...
3965,"[17, 21]","[0, 2]","['69704', '62454', '62549', '9795', '60860']",60860,"['9795', '62454', '62549', '60860', '69704']",4994,0_17,train
3966,"[9, 4]","[5, 22]","['80219', '79413', '64614', '50510', '7217']",7217,"['80219', '50510', '7217', '64614', '79413']",4995,5_9,train
3967,"[3, 7]","[0, 24]","['19992', '30859', '4286', '19796', '74418']",74418,"['19796', '19992', '74418', '30859', '4286']",4996,0_3,train
3968,"[0, 7]","[9, 3]","['65801', '10984', '66807', '15613', '74662']",74662,"['15613', '66807', '74662', '65801', '10984']",4997,0_9,train


In [82]:
pd.concat([train_df, test_df]).sample(frac=1, random_state=42).reset_index(drop=True).to_csv("easy_dataset_mid_train_test.csv", index=False)